## Surrogate vs. Thermochemical library
### *Time evolution analysis*

Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from silx.io.dictdump import dicttoh5, h5todict
from reading import read_dataset
from plotting import time_space_2d

Define inputs

In [ ]:
models = ['surrogate', 'thermolib']
scheme = 'strang'
nb_nodes = 1401
nb_iters = 2200
qoi = ['u','Y_O2','Th','Tv']

Read data

In [ ]:
data = read_dataset(models, scheme, nb_iters, nb_nodes)

Compute error

In [ ]:
def mape(y_true, y_pred):
    return 100 * np.absolute((y_true - y_pred) / y_true)

err = {k: mape(data['thermolib'][k], data['surrogate'][k]) for k in qoi}
for k in qoi:
print(np.amax(err[k]))

In [ ]:
shock_pos = [capture_shock(data[model]['u']) for model in models]
shift_idx = np.diff(shock_pos, axis=0).squeeze()
err = {k: mape(data['thermolib'][k], data['surrogate'][k], shift_idx) for k in qoi}

Plot variables and error

In [ ]:
t = data['thermolib']['t']
x = data['thermolib']['x']
for k in qoi:
    for model in models:
        time_space_2d(
            t,
            x,
            data[model][k],
            tlim=[0., 1.e-5],
            xlim=[0.004, 0.009],
            xshift=-0.005,
            show=True,
            cbar_kwargs={
                'label': k,
                'format': '%.1e'
            }
        )
    time_space_2d(
        t,
        x,
        err[k],
        tlim=[0., 1.e-5],
        xlim=[0.005, 0.009],
        xshift=-0.005,
        show=True,
        cbar_kwargs={
            'label': 'Relative error [%]',
            'format': '%.1f'
        }
    )